# K-means로 최적의 K 찾기와 이미지 분류

`my` 폴더의 이미지를 비슷한 것끼리 자동으로 묶습니다.

1. 이미지 한 장을 숫자 데이터로 바꿉니다.
2. K=2부터 K=12까지 군집화하여 inertia를 비교합니다.
3. 엘보우 그래프를 보고 최종 K를 **6**으로 선택합니다.
4. 6개 군집으로 분류한 이미지를 화면에 출력합니다.

> K-means에는 정답 레이블이 없으므로, K는 그래프와 분류 결과를 함께 보고 정합니다. 여기서는 그래프의 꺾이는 지점과 이미지 결과를 보고 K=6을 사용합니다.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans

# Windows 한글 글꼴 설정: 그래프 제목·축 이름의 한글이 깨지는 것을 막는다.
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 기호가 네모로 보이는 현상 방지

## 1. 이미지 불러오기

K-means는 숫자 배열을 입력으로 받습니다. 따라서 각 JPG 이미지를 흑백으로 바꾸고, 크기를 모두 50×50으로 통일합니다. 이미지 한 장은 2,500개(50×50) 픽셀값으로 이루어진 데이터가 됩니다.

In [ ]:
images = []

# my 폴더 안의 JPG 파일을 하나씩 읽는다.
for file in os.listdir('./my'):
    if file.lower().endswith('.jpg'):
        path = os.path.join('./my', file)

        # L: 흑백 이미지, resize: 모든 이미지의 크기를 동일하게 맞춤
        with Image.open(path) as img:
            img = img.convert('L').resize((50, 50))
            images.append(np.array(img))

images = np.array(images)
print('이미지 배열 모양:', images.shape)  # (이미지 개수, 50, 50)

# (이미지 개수, 50, 50) -> (이미지 개수, 2500)
# K-means가 한 이미지의 2,500개 픽셀값을 하나의 샘플로 인식하게 한다.
fruits_2d = images.reshape(len(images), -1)
print('K-means 입력 모양:', fruits_2d.shape)

## 2. inertia로 K 후보 비교하기

**inertia**는 같은 군집 안의 이미지들이 중심점에서 떨어진 정도입니다. K가 커질수록 inertia는 항상 작아집니다. 따라서 값이 급격히 줄어들다가 완만해지는, 그래프의 **팔꿈치(elbow)** 지점을 찾습니다.

In [ ]:
inertia = []
K_VALUES = range(2, 13)  # K=2, 3, ..., 12를 비교

for k in K_VALUES:
    # random_state를 고정하면 실행할 때마다 같은 결과를 얻는다.
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(fruits_2d)

    # 현재 K일 때의 군집 내 거리 합을 저장한다.
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_VALUES, inertia, marker='o')
plt.xticks(list(K_VALUES))
plt.xlabel('K (군집 개수)')
plt.ylabel('inertia')
plt.title('엘보우 방법으로 K 비교')
plt.grid(alpha=0.3)
plt.show()

## 3. 최종 K 결정

이 데이터에서는 K=6 부근부터 inertia 감소 폭이 비교적 완만해지고, 실제 분류 이미지도 의미 있게 나뉘는 것을 확인했습니다. 따라서 최종 K를 **6**으로 정합니다.

In [ ]:
best_k = 6

plt.figure(figsize=(8, 4))
plt.plot(K_VALUES, inertia, marker='o')
plt.axvline(best_k, color='red', linestyle='--', label=f'선택한 K = {best_k}')
plt.xticks(list(K_VALUES))
plt.xlabel('K (군집 개수)')
plt.ylabel('inertia')
plt.legend()
plt.show()

print('최종 K:', best_k)

## 4. 군집별 이미지 출력 함수

`draw_fruits()`는 한 군집에 들어간 이미지들을 10장씩 격자로 보여 줍니다. 이미지가 너무 많아 화면이 복잡해지지 않도록 각 군집에서 최대 30장만 출력합니다.

In [ ]:
def draw_fruits(arr, max_images=30):
    # 출력할 이미지 수를 최대 max_images장으로 제한한다.
    arr = arr[:max_images]
    n = len(arr)
    rows = int(np.ceil(n / 10))  # 한 줄에 10장씩 배치

    fig, axs = plt.subplots(rows, 10, figsize=(12, rows * 1.3))
    axs = np.ravel(axs)  # 2차원 축 배열을 한 줄로 펼침

    for i in range(n):
        axs[i].imshow(arr[i], cmap='gray')
        axs[i].axis('off')

    # 이미지가 없는 빈 칸은 축을 숨긴다.
    for i in range(n, len(axs)):
        axs[i].axis('off')

    plt.tight_layout()
    plt.show()

## 5. K=6으로 최종 군집화하고 결과 확인

`labels`에는 각 이미지가 어느 군집에 속하는지 0~5의 숫자로 저장됩니다. 예를 들어 `labels == 0`은 0번 군집으로 분류된 이미지만 골라냅니다.

In [ ]:
# 선택한 K=6으로 모든 이미지를 최종 분류한다.
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km.fit_predict(fruits_2d)

# 0번부터 5번까지, 각 군집에 속한 이미지 예시를 출력한다.
for label in range(best_k):
    cluster_images = images[labels == label]
    print(f'군집 {label}: {len(cluster_images)}장')
    draw_fruits(cluster_images)